In [1]:
import pandas as pd

In [4]:
# 데이터 전처리
## 결측치
### 결측치 확인 : isnull().sum()
data = { 'Name' : ['Alice', 'Bob', 'Charlie'],
         'Age' : [25, None, 30],
         'Score' : [90, 85, None]}

df = pd.DataFrame(data)
df.isnull().sum()

Name     0
Age      1
Score    1
dtype: int64

In [5]:
### 결측치 처리 : DataFrame.dropna(axis = 0, how = 'any', subset = None, inplace = False)
#### ex) 결측치가 있는 행 삭제
df_cleaned = df.dropna()
print("결측치 삭제 후 데이터")
print(df_cleaned)


결측치 삭제 후 데이터
    Name   Age  Score
0  Alice  25.0   90.0


In [7]:
### 결측치를 특정 값으로 대체 : DataFrame.fillna(value, inplace = False)

#### ex) 결측치를 평균값으로 대체
mean_age = df['Age'].mean() # 평균값 계산
df['Age'] = df['Age'].fillna(mean_age)
print("Age 열 평균값으로 결측치 대체")
print(df)


Age 열 평균값으로 결측치 대체
      Name   Age  Score
0    Alice  25.0   90.0
1      Bob  27.5   85.0
2  Charlie  30.0    NaN


In [8]:
#### ex) 결측치를 0으로 대체
df['Score'] = df['Score'].fillna(0)
print(df)

      Name   Age  Score
0    Alice  25.0   90.0
1      Bob  27.5   85.0
2  Charlie  30.0    0.0


In [9]:
## 이상치
### 이상치 확인
data_height = {'Height' : [170, 150, 160, 180, 350]}
df = pd.DataFrame(data_height)

print(df)

   Height
0     170
1     150
2     160
3     180
4     350


In [14]:
#### 1.최댓값 , 최솟값으로 이상치 확인
print('max:' , df['Height'].max())
print('min:' , df['Height'].min())

max: 350
min: 150


In [26]:
#### 2. IQR 사용하여 확인하기
Q1 = df['Height'].quantile(0.25)
Q3 = df['Height'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['Height'] < lower_bound) | (df['Height'] > upper_bound)]
print(outliers)

   Height   Z-Score
4     350  1.981985


In [27]:
#### 3. Z-Score을 활용하여 확인하기 
### Z - Score란 데이터가 평균에서 얼마나 떨어져있는지를 표준편차 단위로 나타낸 것이며 절댓값이 3 이상이면 이상치로 간주
from scipy.stats import zscore

df['Z-Score'] = zscore(df['Height'])
print('\n Z-Score \n', df)


 Z-Score 
    Height   Z-Score
0     170 -0.428537
1     150 -0.696373
2     160 -0.562455
3     180 -0.294619
4     350  1.981985


In [28]:
outliers = df[df['Z-Score'].abs() > 3]
print(outliers)

Empty DataFrame
Columns: [Height, Z-Score]
Index: []


In [30]:
### 이상치 처리
df_cleaned = df[(df['Height'] >= lower_bound) & (df['Height'] <= upper_bound)]
print(df_cleaned)

   Height   Z-Score
0     170 -0.428537
1     150 -0.696373
2     160 -0.562455
3     180 -0.294619


In [35]:
### 이상치 특정 값으로 대체하기 
#### ex) 이상치를 중앙값으로 대체하기
median_value = df[(df['Height'] >= lower_bound) & (df['Height'] <= upper_bound)]['Height'].median()
print(median_value)
# df['Height'] = df['Height'].apply(lambda x: median_value if (x < lower_bound or x > upper_bound) else x)
df.loc[(df['Height'] < lower_bound) | (df['Height'] > upper_bound), 'Height'] = median_value
print(df)

165.0
   Height   Z-Score
0   170.0 -0.428537
1   150.0 -0.696373
2   160.0 -0.562455
3   180.0 -0.294619
4   165.0  1.981985


In [39]:
## 수치형 데이터 처리
### 1. 구간화 하기 : pd.cut(x, bins, labels = None, right = True)
#### x : 구간화할 데이터, bins : 구간 분할 기준, lables = 구간 이름, right = 구간의 오른쪽 끝 포함 여부
#### ex) 나이 데이터를 구간화하기
data = {'Age' : [15, 22, 35, 50, 72]}
df = pd.DataFrame(data)

bins = [0, 20, 40, 60, 80]
labels = ['10대', '20대', '30대', '40대 이상']
df['Age_group'] = pd.cut(df['Age'], bins = bins, labels = labels)
print(df)

   Age Age_group
0   15       10대
1   22       20대
2   35       20대
3   50       30대
4   72    40대 이상


In [41]:
#### 자동 구간화하기 : pd.qcut()
df['Age_Quantile'] = pd.qcut(df['Age'], q=3, labels=['하위', '중위', '상위'])
print(df)

   Age Age_group Age_Quantile
0   15       10대           하위
1   22       20대           하위
2   35       20대           중위
3   50       30대           상위
4   72    40대 이상           상위


In [42]:
### 2. 스케일링 변환
#### 스케일링: 수치형 데이터를 일정한 범위로 조정하는 작업
#### 1) 정규화 : 0과 1사이로 변환하는 작업
from sklearn.preprocessing import MinMaxScaler

In [45]:
data = {'Height' : [150, 160, 170, 180, 190]}
df = pd.DataFrame(data)

scaler = MinMaxScaler()
df['Height_Normalized'] = scaler.fit_transform(df[['Height']])
print(df)

   Height  Height_Normalized
0     150               0.00
1     160               0.25
2     170               0.50
3     180               0.75
4     190               1.00


In [ ]:
#### 2) 표준화: 데이터의 평균을 0, 표준편차를 1로 변환하는 잣업
from sklearn.preprocessing import StandardScaler
